# Reduce Operators with Seismic Data Objects
## Overview
In the multiple places in this course we refer to the "map-reduce" paradigm for parallel processing.   None of the workflows used in the course demonstrate the "reduce" concept.   This notebook is a sidebar to the course students can and should explore on their own time. 

Keep in mind before starting that we have not found reduce operators useful for reasons reiterated at the end of this notebook where you can better appreciate the context.

## Generate test data set
First we need to create a simple test data set to illustrate the concepts.  We do that here by generating set of N ensembles with identical start and end times.   The contents are all just constant values but each ensemble has a different number in the members so we can validate the end result.

In [1]:
from mspasspy.ccore.seismic import TimeSeries,TimeSeriesEnsemble,DoubleVector
import numpy as np

# create a template TimeSeries with 100 samples with ones
N=100   # size of each data array for TimeSeries objects
N_ens=3 # number of distinct ensembles to generate
M=5     # number of 
datavector=np.ones(N)
d0 = TimeSeries(N)
d0.dt=0.1
d0.set_live()
d0.data = DoubleVector(datavector)  # needed for type conversion from numpy array
ensembles=list()
for i in range(N_ens):
    di = TimeSeries(d0)
    # i+1 in all cases so we never have zero data or so evid is consistent with value for this test
    di *= float(i+1)
    di["evid"] = i+1
    ens = TimeSeriesEnsemble(M)
    for j in range(M):
        ens.member.append(di)
    ens.set_live()
    ensembles.append(ens)

The ensembles were just a convenient way to generate some data with a common Metadata key ("evid").   To work with dask we have to flatten the data into a simple list.   This simple llittle code section does that putting the results into a simple python list referenced with the symbol "tslist".

In [2]:
# convert to a single big list
tslist = list()
for i in range(3):
    for j in range(M):
        tslist.append(ensembles[i][j])
print("Size of list=",len(tslist))

Size of list= 15


It is important you understand eacho of the components of tslist have the Metadata key "evid" defined.   tslist isn't that long so it is instructive to consider this output:

In [3]:
print("evid  npts  data_vector_value")
for d in tslist:
    print(d["evid"],d.npts,d.data[0])

evid  npts  data_vector_value
1 100 1.0
1 100 1.0
1 100 1.0
1 100 1.0
1 100 1.0
2 100 2.0
2 100 2.0
2 100 2.0
2 100 2.0
2 100 2.0
3 100 3.0
3 100 3.0
3 100 3.0
3 100 3.0
3 100 3.0


## Set up and run reduce operator (dask foldby)
The "reduce" operation requires a "binary operator" that is will combine any two elements of a list like `tslist` and return another of the same type (reduce the size by one).   A very important limitation is that the operation must be "associative".  That is, the function must satisfy `f(a,f(b,c)) = f(f(a,b),c)`.   In seismology a simple "stack" (aka average) operator is a type example that statisfies that property.   

With that background here is a simple stack function for `TimeSeries` data:

In [4]:
def stack(d1,d2):
    """
    None initializer stack operator
    """
    if d1 is None:
        return d2
    else:
        d = TimeSeries(d1)
        d += d2
        return d      

This function is a bit more complex that you might think it needs to be.  The else clause is an implementation issue for TimeSeries that isn't important for this tutorial.  The `if d1 is None:` conditional, however, is fundamental.   A function like this needs an initialization condition.  It can be handled a number of ways, but this is a simple solution for this introductory tutorial.   Where that enters is in this code box that actually runs the reduce (aka dask `foldby` method of "bad") function:

In [5]:
import dask.bag as dbg

b = dbg.from_sequence(tslist)   # convert list to a dask bag
b = b.foldby("evid",
             binop=stack,
             initial=None,        # this is why the stack function has the None conditional
            )
s = b.compute()   # tells task to compute the result

The return from this is a little weird shown by this little code box:

In [6]:
for key,val in s:
    print(key,type(val))

1 <class 'mspasspy.ccore.seismic.TimeSeries'>
2 <class 'mspasspy.ccore.seismic.TimeSeries'>
3 <class 'mspasspy.ccore.seismic.TimeSeries'>


In words s is a list of tuples.  The first component of each tuple is the key used to sort the data being combined.  Here that is the value of "evid" we set earlier.  The second component is the result of the reduce (foldby) operation.  In our case that is a stack of the data.   The following shows this worked as expected for summing data as each TimeSeries we initially created has a data array of constant values equal to the value of "evid".   

In [7]:
for key, val in s:
    evid = key
    expected = float(evid*M)
    print(key,expected,val.data[0])

1 5.0 5.0
2 10.0 10.0
3 15.0 15.0


## A Caveat
After seeing that example, you are more likely to accept the following assertions for why "reduce" in a dask workflow has proven to be not very important for seismology data:
1.  In all cases we have seen the "reduce" sorting implicit in a reduce (FoldBy) is better done by using the MongoDB database.   A known issue with `foldby` is that it is dumb about what data it sends to what worker.   Use of `foldby` on a large data set is very inefficient because it consumes unnecessary time sending data between workers.   A database is very good at producing presorted data to assemble "ensembles" that are to be handled more cleanly than putting all the data into a single container inherent in the use of `foldby`.
2.  Actual use of `foldby` can be a memory pig and it does not handle large seismic data sets well.  It is clear to us the reduce paradigm is more appropriate for large numbers of small pieces of data like single words from documents.  Given it's roots from Google that is not surprising in retrospect.
3.  A large class of potential reduce algorithms don't satisfy the associative rule [see e.g.](https://medium.com/@adityashete009/mapreduce-cef86a82cca9)   
4.  As the above example shows the dask `foldby` function is a nightmare to get right with the combined constraints of proper initialization and use of function objects to define the operators. 